**Generating Policy Sales Data**

In [1]:
import pandas as pd
import numpy as np

# Number of rows
n = 1_000_000

# Random purchase dates in 2024
purchase_dates = pd.to_datetime("2024-01-01") + pd.to_timedelta(
    np.random.randint(0, 365, size=n), unit="D"
)

# Tenure distribution: 20% 1yr, 30% 2yr, 40% 3yr, 10% 4yr
tenures = np.random.choice([1,2,3,4], size=n, p=[0.2,0.3,0.4,0.1])

# Premiums
premiums = tenures * 100

# Start and End dates
start_dates = purchase_dates + pd.to_timedelta(365, unit="D")
end_dates = start_dates + pd.to_timedelta(tenures*365, unit="D")

# Build DataFrame
policy_sales = pd.DataFrame({
    "Customer_ID": np.arange(1, n+1),
    "Vehicle_ID": np.arange(1, n+1),
    "Vehicle_Value": 100000,
    "Policy_Tenure": tenures,
    "Premium": premiums,
    "Policy_Purchase_Date": purchase_dates,
    "Policy_Start_Date": start_dates,
    "Policy_End_Date": end_dates
})

policy_sales.head()

,Customer_ID,Vehicle_ID,Vehicle_Value,Policy_Tenure,Premium,Policy_Purchase_Date,Policy_Start_Date,Policy_End_Date
0,1,1,100000,2,200,2024-09-19,2025-09-19,2027-09-19
1,2,2,100000,1,100,2024-04-30,2025-04-30,2026-04-30
2,3,3,100000,3,300,2024-06-20,2025-06-20,2028-06-19
3,4,4,100000,3,300,2024-03-04,2025-03-04,2028-03-03
4,5,5,100000,3,300,2024-01-07,2025-01-06,2028-01-06


**Generating Claims Data**

In [2]:
# Claims in 2025: 30% of vehicles purchased on 7,14,21,28
mask_2025 = policy_sales["Policy_Purchase_Date"].dt.day.isin([7,14,21,28])
claims_2025 = policy_sales[mask_2025].sample(frac=0.3, random_state=42)
claims_2025 = claims_2025.assign(
    Claim_Amount=10000,
    Claim_Date=claims_2025["Policy_Start_Date"],
    Claim_Type=1
)

# Claims in Jan-Feb 2026: 10% of 4-year policies
mask_2026 = (policy_sales["Policy_Tenure"]==4) & \
            (policy_sales["Policy_Purchase_Date"].dt.month.isin([1,2]))
claims_2026 = policy_sales[mask_2026].sample(frac=0.1, random_state=42)
claims_2026 = claims_2026.assign(
    Claim_Amount=10000,
    Claim_Date=claims_2026["Policy_Start_Date"],
    Claim_Type=2
)

# Combine
claims = pd.concat([claims_2025, claims_2026])
claims.head()

,Customer_ID,Vehicle_ID,Vehicle_Value,Policy_Tenure,Premium,Policy_Purchase_Date,Policy_Start_Date,Policy_End_Date,Claim_Amount,Claim_Date,Claim_Type
452534,452535,452535,100000,2,200,2024-01-07,2025-01-06,2027-01-06,10000,2025-01-06,1
980495,980496,980496,100000,3,300,2024-02-28,2025-02-27,2028-02-27,10000,2025-02-27,1
515806,515807,515807,100000,2,200,2024-01-28,2025-01-27,2027-01-27,10000,2025-01-27,1
771101,771102,771102,100000,2,200,2024-08-21,2025-08-21,2027-08-21,10000,2025-08-21,1
467973,467974,467974,100000,2,200,2024-10-21,2025-10-21,2027-10-21,10000,2025-10-21,1


**Saving it as CSV file**

In [3]:
policy_sales.to_csv("policy_sales.csv", index=False)
claims.to_csv("claims.csv", index=False)